In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)
    
# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /home/fre.gilad/source/AgentDaC/notebooks


## Env

In [2]:
from src.utils.env import prepare_environment

_ = prepare_environment("../api_keys")

INFO 08-09 01:35:09 [env.py:30] Setting API tokens: ['WANDB_API_KEY', 'OPENPIPE_API_KEY', 'OPENAI_API_KEY']
INFO 08-09 01:35:09 [env.py:42] Setting additional variables: {'NCCL_CUMEM_ENABLE': '0', 'VLLM_USE_V1': '0', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn', 'ART_SERVER_TIMEOUT': '300'}


## Model

In [3]:
from src.configs import art_configs
from src.configs import vllm_configs
from src.models import PathConfig

print("Available ART model configurations:")
print(art_configs.available_configs())

print("Available vLLM model configurations:")
print(vllm_configs.available_configs())

base_model = "unsloth/Qwen2.5-14B-Instruct"
project_name = "easy2hard_dac"

path_config = PathConfig(
    base_model=base_model,
    project_name=project_name,
)

Available ART model configurations:
['unsloth/Llama-4-Scout-17B-16E-Instruct', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit', 'unsloth/Qwen2-7B', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen3-14B']
Available vLLM model configurations:
['unsloth/Llama-4-Scout-17B-16E-Instruct', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit', 'unsloth/Qwen2-7B', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen3-14B']


In [4]:
import art

host = "0.0.0.0"
port = 8200

model = art.Model(
    name=base_model,
    project=path_config.project_name,
    inference_api_key=os.getenv("OPENAI_API_KEY", "default"),
    inference_base_url=f"https://{host}:{port}/v1",
)

## Data

In [5]:
from datasets import Dataset, load_dataset, DatasetDict


data: Dataset = load_dataset(
    path="TIGER-Lab/MMLU-Pro",
    split="test",
)  # type: ignore

split_dict = data.train_test_split(test_size=0.3, seed=0)
train_data = split_dict["train"]
test_data = split_dict["test"]

## Inference Clients

In [6]:
from src.vllm_client import VllmClient, VllmRouter

inference_clients: list[VllmClient] = []
vllm_server_ports = [8200]
for port in vllm_server_ports:
    inference_clients.append(VllmClient.from_connection(port=port, base_model=base_model))

vllm_router: VllmRouter = VllmRouter(inference_clients)

## Config 

In [7]:
from experiments.mmlu_pro.trainer import MmluProTrainer
from src.trainer import PromptConfig, StopCriteria

prompt_config = PromptConfig(
    system_root="dac_sys_prompt_gilad_root",
    system_inter="dac_sys_prompt_gilad_inter",
    system_leaf="dac_sys_prompt_gilad_leaf",
)

# prompt_config = PromptConfig(
#     system_root="",
#     system_inter="",
#     system_leaf="",
# )

stop_criteria = StopCriteria(max_depth=1)

trainer = MmluProTrainer(
    model=model,
    vllm_router=vllm_router,
    path_config=path_config,
    prompt_config=prompt_config,
    stop_criteria=stop_criteria,
)

### Inference Test

In [10]:
import random


idx = random.randint(0, len(train_data) - 1)
# idx = 228
print(f"Selected random index: {idx}")
sample = train_data[idx]

print(sample)

problem = sample["question"]
true_answer = sample["answer"]

print(f"Problem: {problem}")
print(f"Answer: {true_answer}")
print("-" * 200)
print()

preds = await trainer.predict([sample], verbose=True, seed=0)

Selected random index: 5673
{'question_id': 3445, 'question': 'The ability of the brain to detect differences in stimulus intensity is best explained by the fact that which of the following varies with the stimulus intensity?', 'options': ['The number of synapses crossed', 'The speed of the action potential', 'The amplitude of the action potential', 'The number of neurons activated', 'The refractory period of the neuron', 'The number of action potentials per second', 'The duration of the action potential', "The permeability of the neuron's membrane", 'The resting potential of the neuron', 'The threshold potential'], 'answer': 'F', 'answer_index': 5, 'cot_content': '', 'category': 'biology', 'src': 'ori_mmlu-college_biology'}
Problem: The ability of the brain to detect differences in stimulus intensity is best explained by the fact that which of the following varies with the stimulus intensity?
Answer: F
-----------------------------------------------------------------------------------

predict:   0%|          | 0/1 [00:00<?, ?it/s]

Role: SYSTEM
Content: You are a highly capable and truthful AI assistant that excels at logical reasoning.

When encountering complex tasks, you may break them down into smaller, manageable sub-tasks. When you do so, these sub-tasks will be assigned to sub-agent to solve and the answer is reported back to you.

Each time you create a sub-task, your turn ends immediately. The sub-agent will then reply with an answer. Then, you automatically regain control for a new turn, free to continue your reasoning with the information received. Importantly, you may engage in multi-round sub-task decomposition across many turns.

In each turn, you may either delegate one additional sub-task based on previous results or provide the final answer if you have enough information.

Single Turn Options:
- You may create a sub-task with <task> </task> block. Your turn ends immediately after the closing </task> marker, and a sub-agent replies with an <answer> </answer> block. You regain control immediately a

## Run Benchmark

In [ ]:
groups = await trainer.rollout(
    train_data.to_list(),
    group_size=1,
    max_exceptions=0.025,
)

rollout:   0%|          | 0/1000 [00:00<?, ?it/s]

In [10]:
from pprint import pprint
from src.utils.analysis import average_metrics

print("Global Metrics:")
all_trajectories = [tr for group in groups for tr in group.trajectories]
pprint(average_metrics(all_trajectories))

print("Answered Metrics:")
answered_trajectories = [tr for tr in all_trajectories if tr.metrics["gave_answer"] == 1]
pprint(average_metrics(answered_trajectories))

print("Unanswered Metrics:")
unanswered_trajectories = [tr for tr in all_trajectories if tr.metrics["gave_answer"] == 0]
pprint(average_metrics(unanswered_trajectories))

Global Metrics:
{'answer_reward': 1.749,
 'behavior_reward': 0.0,
 'completion_tokens': 159.091,
 'direct_calls': 1.0,
 'direct_tasks': 0.0,
 'duration': 21.476251135,
 'format_reward': -0.02,
 'gave_answer': 0.996,
 'is_correct': 0.583,
 'max_depth': 0.0,
 'total_calls': 1.0,
 'total_reward': 1.729,
 'total_tasks': 0.0,
 'total_tokens': 426.709}
Answered Metrics:
{'answer_reward': 1.7560240963855422,
 'behavior_reward': 0.0,
 'completion_tokens': 157.2781124497992,
 'direct_calls': 1.0,
 'direct_tasks': 0.0,
 'duration': 21.40697462550201,
 'format_reward': 0.0,
 'gave_answer': 1.0,
 'is_correct': 0.5853413654618473,
 'max_depth': 0.0,
 'total_calls': 1.0,
 'total_reward': 1.7560240963855422,
 'total_tasks': 0.0,
 'total_tokens': 424.035140562249}
Unanswered Metrics:
{'answer_reward': 0.0,
 'behavior_reward': 0.0,
 'completion_tokens': 610.5,
 'direct_calls': 1.0,
 'direct_tasks': 0.0,
 'duration': 38.726102,
 'format_reward': -5.0,
 'gave_answer': 0.0,
 'is_correct': 0.0,
 'max_depth

In [11]:
from src.utils.analysis import to_dataframe

to_dataframe(all_trajectories).describe()

statistic,problem,answer,agent_answer,category,direct_calls,total_calls,direct_tasks,total_tasks,max_depth,total_tokens,duration,answer_reward,format_reward,behavior_reward,total_reward,is_correct,gave_answer,completion_tokens,reward
str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""1000""","""1000""","""1000""","""1000""",1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0,1000.0
"""null_count""","""0""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,null,null,null,1.0,1.0,0.0,0.0,0.0,426.709,21.476251,1.749,-0.02,0.0,1.729,0.583,0.996,159.091,1.729
"""std""",null,null,null,null,0.0,0.0,0.0,0.0,0.0,290.22122,14.735774,1.479929,0.315753,0.0,1.536203,0.49331,0.063151,253.885578,1.536203
"""min""","""The following is a multiple ch…","""A""","""<h> H </h>""","""biology""",1.0,1.0,0.0,0.0,0.0,110.0,4.658038,0.0,-5.0,0.0,-5.0,0.0,0.0,6.0,-5.0
"""25%""",null,null,null,null,1.0,1.0,0.0,0.0,0.0,214.0,9.182951,0.0,0.0,0.0,0.0,0.0,1.0,7.0,0.0
"""50%""",null,null,null,null,1.0,1.0,0.0,0.0,0.0,315.0,14.748857,3.0,0.0,0.0,3.0,1.0,1.0,7.0,3.0
"""75%""",null,null,null,null,1.0,1.0,0.0,0.0,0.0,568.0,33.944289,3.0,0.0,0.0,3.0,1.0,1.0,279.0,3.0
"""max""","""The following is a multiple ch…","""J""","""To solve this problem, we need…","""psychology""",1.0,1.0,0.0,0.0,0.0,1688.0,56.517309,3.0,0.0,0.0,3.0,1.0,1.0,1345.0,3.0
